# Create a weather information table via web API

In [ ]:
import pandas as pd
import requests
from dotenv import load_dotenv, find_dotenv
import os

Retrieve cities table from MySQL

In [ ]:
load_dotenv(find_dotenv())
schema = os.getenv("SQL_schema")
host = os.getenv("SQL_host")
user = os.getenv("SQL_user")
password = os.getenv("SQL_password")
port = os.getenv("SQL_port")

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

In [76]:
city_df_from_sql = pd.read_sql("cities", con=connection_string)
city_df_from_sql

,city_id,city,country,latitude,longitude,area_km2
0,1,Berlin,Germany,52.52000,13.40500,891.30
1,2,Hamburg,Germany,53.55000,10.00000,755.09
2,3,Frankfurt,Germany,50.11056,8.68222,248.31
3,4,Munich,Germany,48.13750,11.57500,310.71
4,5,Cologne,Germany,50.93639,6.95278,405.15
5,6,Stuttgart,Germany,48.77750,9.18000,207.33
6,7,Düsseldorf,Germany,51.22556,6.77667,217.41
7,8,Leipzig,Germany,51.34000,12.37500,297.80
8,9,Dortmund,Germany,51.51389,7.46528,280.71
9,10,Essen,Germany,51.45083,7.01306,210.34


Function to get the 5 day 3h interval weather forecast of all cities in the cities table

In [ ]:
def get_city_weather_forecast_api(city_df: pd.DataFrame):
    load_dotenv(find_dotenv())
    api_key = os.getenv("weather_api_key")
    data_frame_rows = []

    for city_id in city_df["city_id"]:
        lat = city_df.loc[city_df["city_id"] == city_id, "latitude"].item()
        lon = city_df.loc[city_df["city_id"] == city_id, "longitude"].item()
        weather_request = requests.get(f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={api_key}&units=metric")
        weather_json = weather_request.json()
        for i in range(len(weather_json.get("list"))):
            # timestamp
            json_timestamp = weather_json.get("list", {})[i].get("dt", None)
            timestamp = pd.Timestamp.fromtimestamp(json_timestamp, tz='UTC').tz_convert('Europe/Berlin').tz_localize(None)
            # temperature
            temp = weather_json.get("list", {})[i].get("main", {}).get("temp", None)
            # weather keyword
            weather_keyword = weather_json.get("list", {})[i].get("weather", {})[0].get("main", None)
            # rain probability
            rain_probability = weather_json.get("list", {})[i].get("pop", None)
            # rain volume in mm
            rain_volume = weather_json.get("list", {})[i].get("rain", {}).get("3h", 0.0)
            # snow volume in mm
            snow_volume = weather_json.get("list", {})[i].get("snow", {}).get("3h", 0.0)
            # wind speed in meter/second
            wind_speed = weather_json.get("list", {})[i].get("wind", {}).get("speed", None)
            # data retrieval timestamp
            # data retrieval timestamp
            data_retrieval_time = pd.Timestamp.now(tz='Europe/Berlin').tz_localize(None)
    
            data_frame_rows.append({
                    "city_id": city_id,
                    "timestamp_interval": timestamp,
                    "temperature": temp,
                    "weather_keyword": weather_keyword,
                    "rain_probability": rain_probability,
                    "rain_volume_mm": rain_volume,
                    "snow_volume_mm": snow_volume,
                    "wind_speed_m_s": wind_speed,
                    "data_retrieval_time": data_retrieval_time
                    })

    return pd.DataFrame(data_frame_rows)

Call function to create city_weather_forecast_df

In [78]:
city_weather_forecast_df = get_city_weather_forecast_api(city_df_from_sql)
city_weather_forecast_df

,city_id,timestamp_interval,temperature,weather_keyword,rain_probability,rain_volume_mm,snow_volume_mm,wind_speed_m_s
0,1,2026-05-26 15:00:00+00:00,28.71,Clear,0.00,0.00,0.0,5.10
1,1,2026-05-26 18:00:00+00:00,27.78,Clear,0.00,0.00,0.0,3.77
2,1,2026-05-26 21:00:00+00:00,19.58,Clear,0.00,0.00,0.0,3.82
3,1,2026-05-27 00:00:00+00:00,16.67,Clear,0.00,0.00,0.0,3.72
4,1,2026-05-27 03:00:00+00:00,15.49,Clear,0.00,0.00,0.0,3.62
...,...,...,...,...,...,...,...,...
395,10,2026-05-31 00:00:00+00:00,20.07,Clouds,0.15,0.00,0.0,1.40
396,10,2026-05-31 03:00:00+00:00,19.17,Clouds,0.00,0.00,0.0,1.54
397,10,2026-05-31 06:00:00+00:00,21.12,Rain,0.20,0.45,0.0,2.01
398,10,2026-05-31 09:00:00+00:00,27.88,Clouds,0.00,0.00,0.0,3.75


Insert city_weather_forecast_df into MySQL

In [79]:
city_weather_forecast_df.to_sql('city_weather_forecast',
                  if_exists='append',
                  con=connection_string,
                  index=False)

400

Wrap everything in functions, so it is only necessary to call one function in the end

In [ ]:
"""
OpenWeatherMap forecast fetcher.

Pulls 5-day / 3-hour weather forecasts for all cities stored in the
MySQL 'cities' table and appends the results to 'city_weather_forecast'.
"""

import pandas as pd
import requests
from dotenv import load_dotenv, find_dotenv
import os

def create_connection_string():
    """Build a SQLAlchemy connection string from environment variables.

    Reads SQL_schema, SQL_host, SQL_user, SQL_password, and SQL_port from
    the nearest .env file and returns a mysql+pymysql:// URL.
    """
    load_dotenv(find_dotenv())
    schema = os.getenv("SQL_schema")
    host = os.getenv("SQL_host")
    user = os.getenv("SQL_user")
    password = os.getenv("SQL_password")
    port = os.getenv("SQL_port")

    return f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

def fetch_cities_data(connection_string: str):
    """Load the full 'cities' table, including city_id and coordinates.

    Args:
        connection_string: SQLAlchemy connection URL.

    Returns:
        DataFrame mirroring the 'cities' table schema.
    """
    return pd.read_sql("cities", con=connection_string)

def get_city_weather_forecast_api(city_df: pd.DataFrame):
    """Fetch 3-hour forecast slots for every city via the OpenWeatherMap API.

    Calls a 5-day, 3-hour intervals weather forecast for each
    city using its latitude and longitude from city_df. Timestamps are
    converted from UTC to Europe/Berlin local time (tz-naive) before storage.

    Missing precipitation fields default to 0.0 (rain/snow)so that dry 
    intervals are represented cleanly rather than as NaN.

    Args:
        city_df: DataFrame with at least city_id, latitude, and longitude
                 columns, so the result of fetch_cities_data().

    Returns:
        DataFrame with one row per city per forecast interval and columns:
            city_id, timestamp_interval, temperature, weather_keyword,
            rain_probability, rain_volume_mm, snow_volume_mm,
            wind_speed_m_s, data_retrieval_time.
    """
    load_dotenv(find_dotenv())
    api_key = os.getenv("weather_api_key")
    data_frame_rows = []

    for city_id in city_df["city_id"]:
        lat = city_df.loc[city_df["city_id"] == city_id, "latitude"].item()
        lon = city_df.loc[city_df["city_id"] == city_id, "longitude"].item()
        weather_request = requests.get(f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={api_key}&units=metric")
        weather_json = weather_request.json()
        for i in range(len(weather_json.get("list"))):
            # timestamp
            json_timestamp = weather_json.get("list", {})[i].get("dt", None)
            timestamp = pd.Timestamp.fromtimestamp(json_timestamp, tz='UTC').tz_convert('Europe/Berlin').tz_localize(None)
            # temperature
            temp = weather_json.get("list", {})[i].get("main", {}).get("temp", None)
            # weather keyword
            weather_keyword = weather_json.get("list", {})[i].get("weather", {})[0].get("main", None)
            # rain probability
            rain_probability = weather_json.get("list", {})[i].get("pop", None)
            # rain volume in mm
            rain_volume = weather_json.get("list", {})[i].get("rain", {}).get("3h", 0.0)
            # snow volume in mm
            snow_volume = weather_json.get("list", {})[i].get("snow", {}).get("3h", 0.0)
            # wind speed in meter/second
            wind_speed = weather_json.get("list", {})[i].get("wind", {}).get("speed", None)
            # data retrieval timestamp
            data_retrieval_time = pd.Timestamp.now(tz='Europe/Berlin').tz_localize(None)
    
            data_frame_rows.append({
                    "city_id": city_id,
                    "timestamp_interval": timestamp,
                    "temperature": temp,
                    "weather_keyword": weather_keyword,
                    "rain_probability": rain_probability,
                    "rain_volume_mm": rain_volume,
                    "snow_volume_mm": snow_volume,
                    "wind_speed_m_s": wind_speed,
                    "data_retrieval_time": data_retrieval_time
                    })

    return pd.DataFrame(data_frame_rows)


def store_weather_data(weather_df: pd.DataFrame, connection_string: str):
    """Append forecast rows to the 'city_weather_forecast' table.

    Args:
        weather_df: DataFrame produced by get_city_weather_forecast_api().
        connection_string: SQLAlchemy connection URL.
    """
    weather_df.to_sql('city_weather_forecast',
                  if_exists='append',
                  con=connection_string,
                  index=False)

def create_city_weather_forecast():
    """End-to-end pipeline: fetch forecasts and persist them to MySQL.

    Steps:
        1. Build the DB connection string from .env variables.
        2. Load city coordinates from the 'cities' table.
        3. Retrieve 5-day forecasts from OpenWeatherMap for all cities.
        4. Append the forecast rows to 'city_weather_forecast'.
    """
    connection_string = create_connection_string()
    cities_df = fetch_cities_data(connection_string)
    weather_df = get_city_weather_forecast_api(cities_df)
    store_weather_data(weather_df, connection_string)

In [22]:
create_city_weather_forecast()